# 🌊 Predicción del Nivel de los Ríos (t+1) — Proyecto completo

**Objetivo:** predecir el nivel del río (en metros) del **día siguiente** a partir de los niveles de los días anteriores, y comparar 7 modelos distintos.

Este notebook reúne **todo el flujo del proyecto de principio a fin** y es la versión educativa de los scripts de `src/`. El código de cada modelo es el mismo que el de su archivo `.py`.

## La plantilla de 4 pasos

Todos los modelos siguen la misma estructura:

| Paso | Qué se hace |
|------|-------------|
| **PASO 1: Preparación de datos** | Crear lags, dividir train / val / test, escalar |
| **PASO 2: Definición del modelo** | Elegir el modelo y sus hiperparámetros |
| **PASO 3: Entrenamiento** | Ajustar el modelo con los datos de entrenamiento |
| **PASO 4: Predicción e inversión de escala** | Predecir el test y volver a metros para evaluar |

Cuando un modelo necesita salirse de la plantilla, verás un comentario `# [JUSTIFICACIÓN]: ...` explicando por qué.

## Modelos comparados

| Familia | Modelos | Archivo |
|---------|---------|---------|
| Estadístico | ARIMA | `src/models/arima_model.py` |
| Árboles | Random Forest, XGBoost | `src/models/tree_models.py` |
| Redes neuronales | LSTM, CNN 1D, Transformer | `src/models/neural_models.py` |
| Google | Prophet | `src/models/google_model.py` |

## 0. Imports y configuración

Todas las librerías se importan al principio. Si alguna falta, instálala con `pip install -r requirements.txt`.

Cambia la variable `ESTACION` para analizar otro río: `"Palua"`, `"Orinoco"`, `"Caicara"`, `"Ciudad_Bolivar"` o `"Ayacucho"`.

In [ ]:
import copy
import logging
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from prophet import Prophet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.arima.model import ARIMA
from torch import nn
from xgboost import XGBRegressor

# Silenciamos avisos de convergencia y mensajes internos para que la salida sea legible
warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

# ---------------- Configuración ----------------
RUTA_DATOS = "data/datos_rios.xlsx"
ESTACIONES = ["Palua", "Orinoco", "Caicara", "Ciudad_Bolivar", "Ayacucho"]
ESTACION = "Palua"          # <-- río que vamos a analizar en este notebook

SEMILLA = 42                # semilla fija para resultados reproducibles
NUMERO_DE_LAGS = 7          # usamos los niveles de los 7 días anteriores
PORCENTAJE_TRAIN = 0.70
PORCENTAJE_VAL = 0.15       # el 15% restante es test

random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

plt.rcParams["figure.figsize"] = (14, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Aquí iremos guardando las predicciones (en metros) y las métricas de cada modelo
predicciones = {}
metricas = {}

## 1. Carga de datos e imputación de valores faltantes

1. Leemos el Excel y convertimos `Fecha` en el índice (tipo `datetime`).
2. Forzamos frecuencia diaria: si faltara algún día, aparece como fila vacía (`NaN`).
3. Rellenamos los `NaN` con **interpolación temporal** (`method="time"`): se estima el valor que falta en función de la distancia en días a los valores conocidos.

In [ ]:
df = pd.read_excel(RUTA_DATOS)

df["Fecha"] = pd.to_datetime(df["Fecha"])
df = df.set_index("Fecha")
df = df.sort_index()

fechas_repetidas = df.index.duplicated(keep="first")
df = df[~fechas_repetidas]
df = df.asfreq("D")

print("Valores faltantes ANTES de interpolar:")
print(df.isna().sum())

df = df.interpolate(method="time", limit_direction="both")

print("\nValores faltantes DESPUÉS de interpolar:", int(df.isna().sum().sum()))
df.head()

In [ ]:
fig, ax = plt.subplots()
for estacion in ESTACIONES:
    ax.plot(df.index, df[estacion], label=estacion, linewidth=1)
ax.set_title("Nivel diario de todas las estaciones")
ax.set_ylabel("Nivel (m)")
ax.legend(ncol=5, loc="upper left")
plt.show()

## 2. Preparación común (PASO 1 compartido por todos los modelos)

### 2.1 División cronológica 70 / 15 / 15
En series temporales **nunca** se mezclan los datos al azar: el modelo debe aprender del pasado y ser evaluado en el futuro.

- **Train (70%)**: para entrenar.
- **Validación (15%)**: para ajustar / detener el entrenamiento (*early stopping*).
- **Test (15%)**: datos que el modelo nunca ha visto; aquí se calculan las métricas finales.

### 2.2 Escalado Min-Max
Lleva los niveles al rango [0, 1]. El escalador se ajusta **solo con train** para no "espiar" el futuro (*data leakage*).

### 2.3 Lags (retardos)
Convertimos la serie en una tabla supervisada: para cada día `t`, las entradas son los niveles de `t-1 ... t-7` y la salida es el nivel de `t`.

| Fecha | lag_1 (ayer) | lag_2 | ... | lag_7 | objetivo (hoy) |
|-------|-------|-------|-----|-------|----------|

In [ ]:
serie = df[ESTACION]   # serie en metros (sin escalar)

# --- 2.1) Puntos de corte cronológicos ---
total_dias = len(serie)
indice_fin_train = int(total_dias * PORCENTAJE_TRAIN)
indice_fin_val = int(total_dias * (PORCENTAJE_TRAIN + PORCENTAJE_VAL))
fecha_fin_train = serie.index[indice_fin_train - 1]
fecha_fin_val = serie.index[indice_fin_val - 1]

# --- 2.2) Escalado Min-Max ajustado SOLO con train ---
valores_train = serie.iloc[:indice_fin_train].to_numpy().reshape(-1, 1)
escalador = MinMaxScaler()
escalador.fit(valores_train)

valores_escalados = escalador.transform(serie.to_numpy().reshape(-1, 1))
serie_escalada = pd.Series(valores_escalados.ravel(), index=serie.index)

# --- 2.3) Tabla supervisada con lags ---
tabla = pd.DataFrame(index=serie_escalada.index)
tabla["objetivo"] = serie_escalada
columnas_lags = []
for k in range(1, NUMERO_DE_LAGS + 1):
    tabla["lag_" + str(k)] = serie_escalada.shift(k)
    columnas_lags.append("lag_" + str(k))
tabla = tabla.dropna()   # los primeros 7 días no tienen lags completos

# --- 2.4) Separar train / val / test por fechas ---
tabla_train = tabla[tabla.index <= fecha_fin_train]
tabla_val = tabla[(tabla.index > fecha_fin_train) & (tabla.index <= fecha_fin_val)]
tabla_test = tabla[tabla.index > fecha_fin_val]

x_train = tabla_train[columnas_lags]
y_train = tabla_train["objetivo"]
x_val = tabla_val[columnas_lags]
y_val = tabla_val["objetivo"]
x_test = tabla_test[columnas_lags]
y_test = tabla_test["objetivo"]

fechas_test = tabla_test.index
y_test_metros = serie.loc[fechas_test].to_numpy()   # valores reales para evaluar

print("Train:", len(x_train), "días |", x_train.index.min().date(), "→", x_train.index.max().date())
print("Val:  ", len(x_val), "días |", x_val.index.min().date(), "→", x_val.index.max().date())
print("Test: ", len(x_test), "días |", x_test.index.min().date(), "→", x_test.index.max().date())
tabla_train.head()

Dos funciones auxiliares que usaremos con **todos** los modelos:
- `invertir_escala`: pasa una predicción escalada (0-1) de vuelta a metros.
- `calcular_metricas`: calcula **RMSE**, **MAE** y **R²** en metros.
  - **RMSE** (raíz del error cuadrático medio): penaliza mucho los errores grandes. Menor es mejor.
  - **MAE** (error absoluto medio): error promedio en metros. Menor es mejor.
  - **R²**: qué proporción de la variación del río explica el modelo. 1 es perfecto.

In [ ]:
def invertir_escala(escalador, valores_escalados):
    # Convierte valores escalados (0-1) de vuelta a metros.
    valores_2d = np.asarray(valores_escalados, dtype=float).reshape(-1, 1)
    valores_metros = escalador.inverse_transform(valores_2d)
    return valores_metros.ravel()


def calcular_metricas(y_real, y_pred):
    # Calcula RMSE, MAE y R² (ambos argumentos en metros).
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae = mean_absolute_error(y_real, y_pred)
    r2 = r2_score(y_real, y_pred)
    return {"RMSE": float(rmse), "MAE": float(mae), "R2": float(r2)}


# Comprobación: invertir la escala de y_test debe devolver los valores reales en metros
recuperado = invertir_escala(escalador, y_test)
print("¿La inversión de escala es correcta?", np.allclose(recuperado, y_test_metros))

In [ ]:
fig, ax = plt.subplots()
ax.plot(serie[serie.index <= fecha_fin_train], label="Train (70%)", color="#2563eb")
ax.plot(serie[(serie.index > fecha_fin_train) & (serie.index <= fecha_fin_val)], label="Validación (15%)", color="#f59e0b")
ax.plot(serie[serie.index > fecha_fin_val], label="Test (15%)", color="#16a34a")
ax.set_title("División cronológica de la serie — " + ESTACION)
ax.set_ylabel("Nivel (m)")
ax.legend()
plt.show()

---
## 3. Modelo estadístico: ARIMA

**ARIMA(p, d, q)** es el modelo clásico de series temporales:
- **AR (p)**: el nivel de mañana depende de los `p` niveles anteriores.
- **I (d)**: se trabaja con las diferencias entre días (`d` veces) para quitar la tendencia.
- **MA (q)**: se corrige con los `q` errores de predicción anteriores.

⚠️ ARIMA **no usa** la tabla de lags ni el escalado: trabaja directamente sobre la serie en metros. Elegimos `(p, d, q)` probando combinaciones y quedándonos con la de menor **AIC** (criterio que equilibra ajuste y complejidad).

In [ ]:
# === PASO 1: PREPARACIÓN DE DATOS ===
# [JUSTIFICACIÓN]: ARIMA no usa una matriz X de lags ni un vector y. Trabaja
# directamente sobre la serie temporal completa y en metros (sin escalar),
# porque el propio modelo calcula internamente sus "lags". Solo usamos las
# fechas de corte para respetar el split 70/15/15.
serie_train = serie[serie.index <= fecha_fin_train]
serie_train_val = serie[serie.index <= fecha_fin_val]

# === PASO 2: DEFINICIÓN / CONFIGURACIÓN DEL MODELO ===
# [JUSTIFICACIÓN]: en ARIMA "definir el modelo" significa elegir el orden (p, d, q).
# Probamos varias combinaciones SOLO con train y nos quedamos con la de menor AIC.
VALORES_P = [0, 1, 2, 3]
VALORES_Q = [0, 1, 2]
VALOR_D = 1

mejor_orden = (1, VALOR_D, 0)
mejor_aic = float("inf")
for p in VALORES_P:
    for q in VALORES_Q:
        if p == 0 and q == 0:
            continue
        orden = (p, VALOR_D, q)
        # [JUSTIFICACIÓN]: con algunos órdenes el cálculo interno de ARIMA es
        # numéricamente inestable y lanza un error. Ese candidato simplemente
        # se descarta y se sigue probando el resto (no es un "fallback" de librería).
        try:
            resultado_prueba = ARIMA(serie_train, order=orden).fit()
        except (np.linalg.LinAlgError, ValueError):
            print("  ARIMA", orden, "-> descartado (no converge)")
            continue
        print("  ARIMA", orden, "-> AIC =", round(resultado_prueba.aic, 1))
        if resultado_prueba.aic < mejor_aic:
            mejor_aic = resultado_prueba.aic
            mejor_orden = orden
print("Orden elegido:", mejor_orden)

modelo = ARIMA(serie_train_val, order=mejor_orden)

# === PASO 3: ENTRENAMIENTO ===
# Una vez elegido el orden, se reajustan los parámetros con train + val.
modelo_entrenado = modelo.fit()

# === PASO 4: PREDICCIÓN (YA EN METROS) ===
# [JUSTIFICACIÓN]: no hay que invertir ninguna escala porque ARIMA se entrenó en metros.
# Aplicamos los MISMOS parámetros a la serie completa: así la predicción del día t
# usa solo los datos reales hasta t-1 (sin volver a entrenar).
modelo_serie_completa = modelo_entrenado.apply(serie)
prediccion = modelo_serie_completa.get_prediction(start=fechas_test[0], end=fechas_test[-1])
y_pred = prediccion.predicted_mean.to_numpy()

predicciones["ARIMA"] = y_pred
metricas["ARIMA"] = calcular_metricas(y_test_metros, y_pred)
print("ARIMA:", metricas["ARIMA"])

---
## 4. Modelos de árboles

### 4.1 Random Forest (Bagging)
Entrena **cientos de árboles de decisión independientes**, cada uno con una muestra aleatoria de los datos, y **promedia** sus predicciones. Es robusto y casi no necesita ajuste.

Entradas: `lag_1 ... lag_7` (escalados). Salida: nivel escalado de hoy → se invierte a metros.

In [ ]:
# === PASO 1: PREPARACIÓN DE DATOS ===
# Usamos x_train, y_train y x_test preparados en la sección 2.

# === PASO 2: DEFINICIÓN DEL MODELO ===
modelo = RandomForestRegressor(
    n_estimators=300,       # número de árboles
    max_depth=12,           # profundidad máxima de cada árbol
    min_samples_leaf=2,     # mínimo de muestras en cada hoja
    n_jobs=-1,              # usar todos los núcleos del procesador
    random_state=SEMILLA,
)

# === PASO 3: ENTRENAMIENTO ===
# [JUSTIFICACIÓN]: Random Forest no usa el conjunto de validación porque no tiene
# "early stopping": entrena todos sus árboles de una vez solo con train.
modelo.fit(x_train, y_train)

# === PASO 4: PREDICCIÓN E INVERSIÓN DE ESCALA ===
y_pred_escalado = modelo.predict(x_test)
y_pred = invertir_escala(escalador, y_pred_escalado)

predicciones["RandomForest"] = y_pred
metricas["RandomForest"] = calcular_metricas(y_test_metros, y_pred)
print("RandomForest:", metricas["RandomForest"])

### 4.2 XGBoost (Boosting)
Construye árboles **uno detrás de otro**: cada árbol nuevo intenta corregir los errores de los anteriores. Usamos el conjunto de **validación** para detenernos cuando deja de mejorar (*early stopping*).

In [ ]:
# === PASO 1: PREPARACIÓN DE DATOS ===
# Usamos x_train, y_train, x_val, y_val y x_test preparados en la sección 2.

# === PASO 2: DEFINICIÓN DEL MODELO ===
modelo = XGBRegressor(
    n_estimators=1000,          # máximo de árboles (se detiene antes si no mejora)
    learning_rate=0.03,         # cuánto corrige cada árbol nuevo
    max_depth=5,
    subsample=0.9,              # % de filas que ve cada árbol
    colsample_bytree=0.9,       # % de columnas (lags) que ve cada árbol
    early_stopping_rounds=50,   # parar si 50 árboles seguidos no mejoran en val
    random_state=SEMILLA,
    n_jobs=-1,
)

# === PASO 3: ENTRENAMIENTO ===
# [JUSTIFICACIÓN]: a diferencia de Random Forest, XGBoost recibe el conjunto de
# validación (eval_set) para aplicar "early stopping" y evitar el sobreajuste.
modelo.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
print("Árboles usados (mejor iteración):", modelo.best_iteration + 1)

# === PASO 4: PREDICCIÓN E INVERSIÓN DE ESCALA ===
y_pred_escalado = modelo.predict(x_test)
y_pred = invertir_escala(escalador, y_pred_escalado)

predicciones["XGBoost"] = y_pred
metricas["XGBoost"] = calcular_metricas(y_test_metros, y_pred)
print("XGBoost:", metricas["XGBoost"])

---
## 5. Redes neuronales (PyTorch)

Las tres redes reciben **ventanas de 7 días** con forma `(muestras, 7 pasos, 1 variable)` y devuelven el nivel escalado del día siguiente.

| Red | Idea principal |
|-----|----------------|
| **LSTM** | Red recurrente con "memoria": lee los 7 días en orden y recuerda lo importante. |
| **CNN 1D** | Filtros que se deslizan sobre la ventana detectando patrones locales (subidas, bajadas). `Conv1D` + `MaxPooling1D`. |
| **Transformer** | *Self-attention*: cada día "mira" a los demás y decide cuánto peso darles. |

En PyTorch cada red es una **clase** con dos partes:
- `__init__`: qué capas tiene.
- `forward`: en qué orden pasan los datos por esas capas.

In [ ]:
class RedLSTM(nn.Module):
    # LSTM de 2 capas seguida de capas densas.

    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, num_layers=2,
                            batch_first=True, dropout=0.1)
        self.capa_densa = nn.Linear(64, 32)
        self.activacion = nn.ReLU()
        self.capa_salida = nn.Linear(32, 1)

    def forward(self, x):
        # x: (lote, 7 pasos, 1)
        salidas_lstm, _ = self.lstm(x)              # (lote, 7, 64)
        ultimo_paso = salidas_lstm[:, -1, :]        # nos quedamos con el último día: (lote, 64)
        h = self.activacion(self.capa_densa(ultimo_paso))
        y = self.capa_salida(h)                     # (lote, 1)
        return y.view(-1)                           # (lote,)


class RedCNN1D(nn.Module):
    # Dos capas Conv1D + MaxPooling1D seguidas de capas densas.

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.pooling = nn.MaxPool1d(kernel_size=2)  # 7 pasos -> 3 pasos
        self.activacion = nn.ReLU()
        self.capa_densa = nn.Linear(64 * 3, 32)
        self.capa_salida = nn.Linear(32, 1)

    def forward(self, x):
        # Conv1d espera (lote, canales, pasos), así que intercambiamos los ejes 1 y 2
        x = x.permute(0, 2, 1)                      # (lote, 1, 7)
        h = self.activacion(self.conv1(x))          # (lote, 64, 7)
        h = self.activacion(self.conv2(h))          # (lote, 64, 7)
        h = self.pooling(h)                         # (lote, 64, 3)
        h = h.reshape(h.shape[0], -1)               # aplanar: (lote, 192)
        h = self.activacion(self.capa_densa(h))
        y = self.capa_salida(h)
        return y.view(-1)


class RedTransformer(nn.Module):
    # Encoder Transformer (self-attention) con posición aprendida para cada día.

    def __init__(self):
        super().__init__()
        dimension = 32
        self.embedding = nn.Linear(1, dimension)    # cada nivel -> vector de 32 números
        # Un vector aprendido por posición para que la red sepa qué día es cada dato
        self.posicion = nn.Parameter(torch.zeros(1, NUMERO_DE_LAGS, dimension))
        capa_encoder = nn.TransformerEncoderLayer(d_model=dimension, nhead=4,
                                                  dim_feedforward=64, dropout=0.1,
                                                  batch_first=True)
        self.encoder = nn.TransformerEncoder(capa_encoder, num_layers=2)
        self.capa_densa = nn.Linear(dimension, 32)
        self.activacion = nn.ReLU()
        self.capa_salida = nn.Linear(32, 1)

    def forward(self, x):
        h = self.embedding(x) + self.posicion       # (lote, 7, 32)
        h = self.encoder(h)                         # (lote, 7, 32)
        ultimo_paso = h[:, -1, :]                   # (lote, 32)
        h = self.activacion(self.capa_densa(ultimo_paso))
        y = self.capa_salida(h)
        return y.view(-1)

### Entrenamiento de las tres redes

El código es el mismo para las tres; solo cambia la clase elegida en el **PASO 2**.

En el **PASO 3** verás un bucle `for` de épocas: en PyTorch el entrenamiento se escribe a mano (no existe un `.fit()` como en scikit-learn). En cada época:
1. Se recorren los datos de train en **lotes** de 32 y se actualizan los pesos.
2. Se mide el error en **validación**.
3. Si la validación no mejora durante 20 épocas seguidas, se para (*early stopping*) y se recuperan los mejores pesos.

In [ ]:
# Hiperparámetros de entrenamiento comunes a las tres redes
EPOCAS_MAXIMAS = 200     # vueltas completas máximas sobre los datos de entrenamiento
TAMANO_LOTE = 32         # cuántas muestras se usan en cada actualización de pesos
TASA_APRENDIZAJE = 0.001
PACIENCIA = 20           # épocas sin mejorar en validación antes de detenerse
NOMBRES_REDES = ["LSTM", "CNN1D", "Transformer"]


def lags_a_tensor(x_lags):
    # Convierte la tabla de lags en un tensor (muestras, 7 pasos, 1) en orden cronológico.
    # Las columnas vienen como lag_1 (ayer) ... lag_7 (hace una semana), pero las
    # redes secuenciales deben leer del pasado al presente: lag_7, ..., lag_1.
    columnas_cronologicas = []
    for k in range(NUMERO_DE_LAGS, 0, -1):
        columnas_cronologicas.append("lag_" + str(k))
    matriz = x_lags[columnas_cronologicas].to_numpy(dtype=np.float32)
    matriz = matriz.reshape(len(matriz), NUMERO_DE_LAGS, 1)
    return torch.tensor(matriz)


historial_val = {}   # error de validación por época (para graficarlo después)

for nombre_red in NOMBRES_REDES:
    # Semillas fijas: misma inicialización de pesos y mismo orden de lotes
    random.seed(SEMILLA)
    np.random.seed(SEMILLA)
    torch.manual_seed(SEMILLA)

    # === PASO 1: PREPARACIÓN DE DATOS ===
    # [JUSTIFICACIÓN]: PyTorch trabaja con tensores, no con DataFrames. Además las redes
    # secuenciales esperan 3 dimensiones: (muestras, pasos de tiempo, variables).
    x_train_t = lags_a_tensor(x_train)
    y_train_t = torch.tensor(y_train.to_numpy(dtype=np.float32))
    x_val_t = lags_a_tensor(x_val)
    y_val_t = torch.tensor(y_val.to_numpy(dtype=np.float32))
    x_test_t = lags_a_tensor(x_test)

    # === PASO 2: DEFINICIÓN DEL MODELO ===
    if nombre_red == "LSTM":
        modelo = RedLSTM()
    elif nombre_red == "CNN1D":
        modelo = RedCNN1D()
    else:
        modelo = RedTransformer()

    funcion_perdida = nn.MSELoss()   # error cuadrático medio
    optimizador = torch.optim.Adam(modelo.parameters(), lr=TASA_APRENDIZAJE)

    # === PASO 3: ENTRENAMIENTO ===
    # [JUSTIFICACIÓN]: a diferencia de scikit-learn (un simple .fit()), en PyTorch el
    # entrenamiento se escribe a mano con un bucle de épocas + early stopping.
    # Barajamos el orden de las VENTANAS (no la serie): cada ventana ya contiene sus
    # 7 días en orden, así que no se rompe la cronología ni hay fuga de información.
    generador = torch.Generator().manual_seed(SEMILLA)
    numero_muestras = len(x_train_t)
    mejor_perdida_val = float("inf")
    mejores_pesos = copy.deepcopy(modelo.state_dict())
    epocas_sin_mejora = 0
    historial_val[nombre_red] = []

    for epoca in range(EPOCAS_MAXIMAS):
        # 3.1) Entrenar una época completa, lote a lote
        modelo.train()
        orden_aleatorio = torch.randperm(numero_muestras, generator=generador)
        for inicio in range(0, numero_muestras, TAMANO_LOTE):
            indices_lote = orden_aleatorio[inicio:inicio + TAMANO_LOTE]
            x_lote = x_train_t[indices_lote]
            y_lote = y_train_t[indices_lote]

            optimizador.zero_grad()                  # borrar gradientes anteriores
            y_pred_lote = modelo(x_lote)             # predicción
            perdida = funcion_perdida(y_pred_lote, y_lote)
            perdida.backward()                       # calcular gradientes
            optimizador.step()                       # actualizar pesos

        # 3.2) Medir el error en validación (sin actualizar pesos)
        modelo.eval()
        with torch.no_grad():
            perdida_val = funcion_perdida(modelo(x_val_t), y_val_t).item()
        historial_val[nombre_red].append(perdida_val)

        # 3.3) Early stopping
        if perdida_val < mejor_perdida_val:
            mejor_perdida_val = perdida_val
            mejores_pesos = copy.deepcopy(modelo.state_dict())
            epocas_sin_mejora = 0
        else:
            epocas_sin_mejora = epocas_sin_mejora + 1
            if epocas_sin_mejora >= PACIENCIA:
                break

    modelo.load_state_dict(mejores_pesos)   # nos quedamos con la mejor época
    print(nombre_red, "-> épocas entrenadas:", epoca + 1, "| mejor MSE val:", round(mejor_perdida_val, 6))

    # === PASO 4: PREDICCIÓN E INVERSIÓN DE ESCALA ===
    modelo.eval()
    with torch.no_grad():
        y_pred_escalado = modelo(x_test_t).numpy()
    y_pred = invertir_escala(escalador, y_pred_escalado)

    predicciones[nombre_red] = y_pred
    metricas[nombre_red] = calcular_metricas(y_test_metros, y_pred)
    print(nombre_red + ":", metricas[nombre_red])

In [ ]:
fig, ax = plt.subplots()
for nombre_red in NOMBRES_REDES:
    ax.plot(historial_val[nombre_red], label=nombre_red)
ax.set_yscale("log")
ax.set_title("Error de validación (MSE) por época — el entrenamiento se detiene al estancarse")
ax.set_xlabel("Época")
ax.set_ylabel("MSE validación (escala log)")
ax.legend()
plt.show()

---
## 6. Modelo de Google: Prophet

**Prophet** (desarrollado por Meta/Facebook y muy usado junto a herramientas de Google Cloud) descompone la serie en:
- **Tendencia** a largo plazo.
- **Estacionalidad anual** (crecida en lluvias, estiaje en sequía).
- **Regresores externos**: le añadimos el nivel real del día anterior (`lag_1`) para que prediga a 1 día vista como los demás modelos.

⚠️ Prophet exige un formato especial: un DataFrame con columnas `ds` (fecha) e `y` (valor).

In [ ]:
# === PASO 1: PREPARACIÓN DE DATOS ===
# [JUSTIFICACIÓN]: Prophet no acepta matrices X / y. Exige un DataFrame con dos
# columnas obligatorias: "ds" (fecha) e "y" (valor a predecir), más las columnas
# de regresores extra. Trabaja en metros, por lo que no se escala.
tabla_prophet = pd.DataFrame()
tabla_prophet["ds"] = serie.index
tabla_prophet["y"] = serie.to_numpy()
tabla_prophet["lag_1"] = serie.shift(1).to_numpy()   # nivel real del día anterior
tabla_prophet = tabla_prophet.dropna()                # el primer día no tiene lag_1

# Prophet no usa early stopping, así que entrena con train + val juntos
tabla_entrenamiento = tabla_prophet[tabla_prophet["ds"] <= fecha_fin_val]
tabla_test_prophet = tabla_prophet[tabla_prophet["ds"].isin(fechas_test)]

# === PASO 2: DEFINICIÓN DEL MODELO ===
modelo = Prophet(
    yearly_seasonality=True,        # ciclo anual de crecida / estiaje
    weekly_seasonality=False,       # el río no depende del día de la semana
    daily_seasonality=False,        # tenemos un dato por día, no por hora
    changepoint_prior_scale=0.05,   # flexibilidad de la tendencia
)
modelo.add_regressor("lag_1")

# === PASO 3: ENTRENAMIENTO ===
modelo.fit(tabla_entrenamiento)

# === PASO 4: PREDICCIÓN (YA EN METROS) ===
# [JUSTIFICACIÓN]: no hay que invertir ninguna escala porque Prophet se entrenó en metros.
prediccion = modelo.predict(tabla_test_prophet[["ds", "lag_1"]])
y_pred = prediccion["yhat"].to_numpy()   # "yhat" es la predicción central de Prophet

predicciones["Prophet"] = y_pred
metricas["Prophet"] = calcular_metricas(y_test_metros, y_pred)
print("Prophet:", metricas["Prophet"])

---
## 7. Comparación de resultados

### 7.1 Tabla de métricas (test, en metros)
Ordenada de mejor a peor según **RMSE**.

In [ ]:
tabla_metricas = pd.DataFrame(metricas).T
tabla_metricas = tabla_metricas.sort_values("RMSE")
tabla_metricas.index.name = "Modelo"
tabla_metricas.style.format("{:.4f}").highlight_min(subset=["RMSE", "MAE"], color="#bbf7d0").highlight_max(subset=["R2"], color="#bbf7d0")

### 7.2 Real vs. predicciones en todo el periodo de test

In [ ]:
COLORES = {
    "ARIMA": "#f59e0b", "RandomForest": "#22c55e", "XGBoost": "#0d9488",
    "LSTM": "#8b5cf6", "CNN1D": "#ec4899", "Transformer": "#3b82f6", "Prophet": "#f97316",
}

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(fechas_test, y_test_metros, color="black", linewidth=2.5, label="Nivel real")
for nombre_modelo in predicciones:
    ax.plot(fechas_test, predicciones[nombre_modelo], linewidth=1, linestyle="--",
            color=COLORES[nombre_modelo], label=nombre_modelo)
ax.set_title("Nivel real vs. predicciones (t+1) — " + ESTACION)
ax.set_ylabel("Nivel (m)")
ax.legend(ncol=4, loc="upper left")
plt.show()

### 7.3 Zoom: últimos 60 días de test

In [ ]:
DIAS_ZOOM = 60

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(fechas_test[-DIAS_ZOOM:], y_test_metros[-DIAS_ZOOM:], color="black",
        linewidth=2.5, marker="o", markersize=3, label="Nivel real")
for nombre_modelo in predicciones:
    ax.plot(fechas_test[-DIAS_ZOOM:], predicciones[nombre_modelo][-DIAS_ZOOM:], linewidth=1.2,
            linestyle="--", color=COLORES[nombre_modelo], label=nombre_modelo)
ax.set_title("Zoom de los últimos " + str(DIAS_ZOOM) + " días — " + ESTACION)
ax.set_ylabel("Nivel (m)")
ax.legend(ncol=4, loc="upper left")
plt.show()

### 7.4 Cada modelo por separado

In [ ]:
nombres_modelos = list(predicciones.keys())
fig, ejes = plt.subplots(len(nombres_modelos), 1, figsize=(14, 2.6 * len(nombres_modelos)), sharex=True)

for i in range(len(nombres_modelos)):
    nombre_modelo = nombres_modelos[i]
    eje = ejes[i]
    eje.plot(fechas_test, y_test_metros, color="black", linewidth=1.5, label="Real")
    eje.plot(fechas_test, predicciones[nombre_modelo], color=COLORES[nombre_modelo],
             linewidth=1.2, label=nombre_modelo)
    rmse = metricas[nombre_modelo]["RMSE"]
    eje.set_title(nombre_modelo + "  (RMSE = " + str(round(rmse, 3)) + " m)", loc="left", fontsize=10)
    eje.set_ylabel("m")
    eje.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

### 7.5 Comparación de errores

In [ ]:
fig, (eje_rmse, eje_mae) = plt.subplots(1, 2, figsize=(14, 4.5))
colores_barras = []
for nombre_modelo in tabla_metricas.index:
    colores_barras.append(COLORES[nombre_modelo])

eje_rmse.barh(tabla_metricas.index, tabla_metricas["RMSE"], color=colores_barras)
eje_rmse.invert_yaxis()
eje_rmse.set_title("RMSE (m) — menor es mejor")

eje_mae.barh(tabla_metricas.index, tabla_metricas["MAE"], color=colores_barras)
eje_mae.invert_yaxis()
eje_mae.set_title("MAE (m) — menor es mejor")

plt.tight_layout()
plt.show()

---
## 8. Guardar resultados para el dashboard

Guardamos las métricas y predicciones de esta estación en `outputs/` con las mismas funciones que usa `main.py`. Así el dashboard (`python -m streamlit run app.py`) puede mostrarlas.

> Para entrenar **todas** las estaciones de una vez, ejecuta en la terminal: `python main.py`

In [ ]:
from src.evaluate import guardar_metricas, guardar_predicciones

for nombre_modelo in predicciones:
    guardar_metricas(ESTACION, nombre_modelo, metricas[nombre_modelo])
    guardar_predicciones(ESTACION, nombre_modelo, fechas_test, y_test_metros, predicciones[nombre_modelo])

print("Resultados de", ESTACION, "guardados en outputs/metrics.json y outputs/predictions.csv")

## Conclusiones

- Todos los modelos siguen la **misma plantilla de 4 pasos**, lo que facilita compararlos y añadir nuevos.
- Las diferencias con la plantilla están justificadas en el código con `# [JUSTIFICACIÓN]`:
  - **ARIMA** y **Prophet** trabajan sobre la serie en metros (sin tabla X/y ni escalado).
  - **XGBoost** y las **redes neuronales** usan validación para el *early stopping*.
  - Las **redes neuronales** necesitan tensores 3D y un bucle de épocas escrito a mano.
- Para añadir un modelo nuevo: copia la celda de un modelo parecido, cambia el **PASO 2** y guarda el resultado en `predicciones` y `metricas`.